In [1]:
# import everything that is necessary

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import sent_tokenize


In [49]:
import torch
from torch import nn

In [85]:
import numpy as np 

In [51]:
d_model = 384  # same as your sentence embedding size
nhead = 8
dim_feedforward = 4 * d_model  # standard rule of thumb

In [3]:
# sentence encoder
sentence_encoder_model = SentenceTransformer('all-MiniLM-L6-v2')

In [4]:
# importing my package 
# importing the path lib
from oral_history_repository_root.util import stream
from pathlib import Path

In [5]:
# test file 
relative_path = Path("..")/"oral_history_repository_project"/"oral_history_repository_root"/"corpus_text"
filename = "corpus1.txt"

In [57]:
batched_sentence_gen = stream.stream_sentence_batch(filename , relative_path , 32)

In [59]:
# this embedding list will hold the embeddings for each sentence in every batch
embedding_list = []

In [61]:
for batch in batched_sentence_gen:
    embedding_list.append(sentence_encoder_model.encode(batch))

In [79]:
# for x in embedding_list[0]:
#     print(x)
# shape(embedding_list[0][0])
type(embedding_list[0][0])
embedding_list[0][0].shape

(384,)

In [53]:

encoder_layer = nn.TransformerEncoderLayer(
d_model=d_model,
nhead=nhead,
dim_feedforward=dim_feedforward,
dropout=0.1,
activation='relu',
batch_first=False  # Transformer expects (seq_len, batch, d_model)
)

In [71]:
num_layers = 3  # number of attention "levels"
transformer_encoder = nn.TransformerEncoder(
    encoder_layer,
    num_layers=num_layers
)


C:\Users\karim\python_workspace\oral_history_repository\Oral_History_Repository\oral_history_repository_virtual_environment\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [81]:
sentence_embedding_list = []
for batch in embedding_list:
    for sentence in batch:
        sentence_embedding_list.append(sentence)

In [83]:
len(sentence_embedding_list)

68

In [89]:
sentence_embedding_np_array = np.array(sentence_embedding_list)

In [95]:
sentence_embedding_tensor = torch.tensor(sentence_embedding_np_array)

In [97]:
sentence_embedding_tensor.shape

torch.Size([68, 384])

In [99]:
sentence_embedding_tensor = sentence_embedding_tensor.unsqueeze(1)

In [101]:
sentence_embedding_tensor.shape

torch.Size([68, 1, 384])

In [133]:
output = transformer_encoder(sentence_embedding_tensor)

In [135]:
output.shape

torch.Size([68, 1, 384])

In [137]:
doc_embedding = output.mean(dim=0).squeeze(0)

In [115]:
# Test query — pass directly as string, not list
test_sentence = "kidnappings in montreal"
test_sentence_encoding = sentence_encoder_model.encode(test_sentence)

In [141]:

doc_embedding = doc_embedding.detach().numpy()

In [143]:
similarity = cosine_similarity([doc_embedding], [test_sentence_encoding])

In [145]:
similarity

array([[0.28737262]], dtype=float32)

In [149]:
# need to prepare, a pipeline so that i can feed documents
# each document will be represented by a corpus 
# i need to know number of lines in the corpus 
# if i know the number of lines
# i can feed them into batches 